In [ ]:
import torch
from datasets import load_dataset

from config import SEED, VALIDATION_SIZE, BPE_VOCAB_SIZE, BLEU_MAX_EXAMPLES
from model.transformer import Transformer
from data.bpe_tokenizers import load_bpe_tokenizer
from inference.greedy import greedy_decode_bpe
from inference.beam import beam_search_decode_bpe
from training.metrics import evaluate_bleu_bpe
import torch
f

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


BPE_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "transformer-translation/tokenizers/"
    "bpe_shared_8k.model"
)

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/transformer-translation/checkpoints/"
    "bpe8000_d128_h8_enc4_dec4_ff512_bs32_lr0.00075initdefaultschedconstanttieFalseb10.9_b20.98_eps1e-09.pt"
)

In [ ]:
tokenizer = load_bpe_tokenizer(
    BPE_MODEL_PATH
)

vocab_size = tokenizer.get_piece_size()

print("BPE vocab size:", vocab_size)


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded.")
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])
print("Perplexity:", checkpoint["perplexity"])

In [ ]:
config = checkpoint["config"]

model = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=config["d_model"],
    num_heads=config["num_heads"],
    d_ff=config["d_ff"],
    num_encoder_layers=config["num_encoder_layers"],
    num_decoder_layers=config["num_decoder_layers"],
    dropout=config["dropout"],
    weight_tying=config.get(
        "weight_tying",
        False
    )
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Model loaded successfully.")
print("Model is in evaluation mode:", not model.training)

In [ ]:
MAX_LEN = 80
BEAM_SIZE = 4
LENGTH_PENALTY_ALPHA = 1.0

print("MAX_LEN:", MAX_LEN)
print("BEAM_SIZE:", BEAM_SIZE)
print("ALPHA:", LENGTH_PENALTY_ALPHA)

In [ ]:
greedy_bleu = evaluate_bleu_bpe(
    model=model,
    dataset_subset=val_dataset,
    tokenizer=tokenizer,
    translate_fn=greedy_decode_bpe,
    method="greedy",
    max_examples=BLEU_MAX_EXAMPLES,
    max_len=MAX_LEN,
    beam_size=BEAM_SIZE,
    alpha=LENGTH_PENALTY_ALPHA
)

print("Greedy BLEU:", greedy_bleu)

In [ ]:
beam_bleu = evaluate_bleu_bpe(
    model=model,
    dataset_subset=val_dataset,
    tokenizer=tokenizer,
    translate_fn=beam_search_decode_bpe,
    method="beam",
    max_examples=BLEU_MAX_EXAMPLES,
    max_len=MAX_LEN,
    beam_size=BEAM_SIZE,
    alpha=LENGTH_PENALTY_ALPHA
)

print("Beam BLEU:", beam_bleu)

In [ ]:
NUM_EXAMPLES = 20

rows = []

for i in range(NUM_EXAMPLES):

    example = val_dataset[i]["translation"]

    german = example["de"]
    reference = example["en"]

    greedy_output = greedy_decode_bpe(
        model=model,
        sentence=german,
        tokenizer=tokenizer,
        device=device,
        max_len=MAX_LEN
    )

    beam_output = beam_search_decode_bpe(
        model=model,
        sentence=german,
        tokenizer=tokenizer,
        device=device,
        max_len=MAX_LEN,
        beam_size=BEAM_SIZE,
        alpha=LENGTH_PENALTY_ALPHA
    )

    rows.append({
        "German Source": german,
        "Reference English": reference,
        "Greedy Output": greedy_output,
        "Beam Output": beam_output,

        # Manual analysis columns
        "Meaning Preserved": "",
        "Fluent English": "",
        "Repetition": "",
        "Too Short": "",
        "Too Generic": "",
        "Main Error Type": ""
    })

error_df = pd.DataFrame(rows)

error_df

In [ ]:
labels = [
    ["Partial", "Mostly", "Yes", "No",  "Yes", "Semantic drift"],
    ["Partial", "Mostly", "No",  "No",  "No",  "Semantic drift"],
    ["No",      "Mostly", "No",  "No",  "Yes", "Semantic drift"],
    ["No",      "Mostly", "Yes", "No",  "Yes", "Repetition"],
    ["No",      "Yes",    "No",  "No",  "Yes", "Semantic drift"],
    ["No",      "Mostly", "No",  "No",  "Yes", "Semantic drift"],
    ["Partial", "Mostly", "Yes", "Yes", "Yes", "Repetition / Omission"],
    ["Partial", "Mostly", "Yes", "Yes", "No",  "Repetition / Omission"],
    ["No",      "Mostly", "No",  "No",  "Yes", "Semantic drift"],
    ["No",      "Mostly", "No",  "No",  "Yes", "Semantic drift"],
    ["Partial", "Mostly", "No",  "Yes", "Yes", "Omission"],
    ["Partial", "Mostly", "Yes", "No",  "Yes", "Repetition"],
    ["No",      "Mostly", "No",  "No",  "Yes", "Semantic drift"],
    ["Partial", "Mostly", "Yes", "No",  "Yes", "Semantic drift"],
    ["No",      "Mostly", "Yes", "No",  "No",  "Semantic drift"],
    ["No",      "Mostly", "No",  "No",  "Yes", "Semantic drift"],
    ["No",      "Mostly", "No",  "No",  "Yes", "Semantic drift"],
    ["Partial", "Mostly", "No",  "Yes", "No",  "Omission"],
    ["Yes",     "Yes",    "No",  "No",  "No",  "Good translation"],
    ["Partial", "Yes",    "Yes", "Yes", "No",  "Omission / Repetition"]
]

columns = [
    "Meaning Preserved",
    "Fluent English",
    "Repetition",
    "Too Short",
    "Too Generic",
    "Main Error Type"
]

for i, values in enumerate(labels):
    error_df.loc[i, columns] = values

In [ ]:
print("Meaning preservation:")
print(error_df.iloc[:20]["Meaning Preserved"].value_counts())

print("\nFluency:")
print(error_df.iloc[:20]["Fluent English"].value_counts())

print("\nRepetition:")
print(error_df.iloc[:20]["Repetition"].value_counts())

print("\nToo generic:")
print(error_df.iloc[:20]["Too Generic"].value_counts())

print("\nMain error types:")
print(error_df.iloc[:20]["Main Error Type"].value_counts())